In [ ]:
import pandas as pd
import numpy as np
import pyarrow

df = pd.read_parquet('../data/processed/base_orders.parquet')

Creating Historical Features for Repeat Customers

In [ ]:
df['num_prior_orders'] = df.groupby('customer_unique_id').cumcount()
df['prev_order_date'] = df.groupby('customer_unique_id')['order_purchase_timestamp'].shift(1)
df['days_since_last_order'] = ( df['order_purchase_timestamp'] - df['prev_order_date']).dt.days
first_order = df.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min')

df['customer_tenure_days'] = (
    df['order_purchase_timestamp'] - first_order
).dt.days
customer_cum_spend = df.groupby('customer_unique_id')['payment_value'].cumsum()
df['prior_total_spend'] = customer_cum_spend.groupby(df['customer_unique_id']).shift(1)
df['avg_prior_order_value'] = (
    df['prior_total_spend'] / df['num_prior_orders']
)

In [ ]:
df['num_prior_orders'].max()

np.int64(14)

Creating the Difference In Expectation for Delivery Date

In [ ]:
df['late_delivery'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype(int)
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

Churn Assignment

In [ ]:
df['next_order_date'] = (
    df.groupby('customer_unique_id')['order_purchase_timestamp']
    .shift(-1)
)
df['churn_90d'] = (
    (df['next_order_date'].isna()) |
    ((df['next_order_date'] - df['order_delivered_customer_date']).dt.days > 90)
).astype(int)

Data Validation

In [ ]:
df = df[df['num_prior_orders'] >= 1]
df = df[
    df['order_delivered_customer_date'].notna() &
    df['order_estimated_delivery_date'].notna()
]

In [ ]:
max_date = df['order_delivered_customer_date'].max()

df = df[
    df['order_delivered_customer_date'] + pd.Timedelta(days=180) <= max_date
]

In [ ]:
df[['late_delivery', 'churn_90d']].mean()

late_delivery    0.053842
churn_90d        0.901851
dtype: float64

In [ ]:
len(df.columns), df.columns

(17,
 Index(['order_id', 'customer_unique_id', 'order_purchase_timestamp',
        'order_delivered_customer_date', 'order_estimated_delivery_date',
        'review_score', 'payment_value', 'num_prior_orders', 'prev_order_date',
        'days_since_last_order', 'customer_tenure_days', 'prior_total_spend',
        'avg_prior_order_value', 'late_delivery', 'delivery_delay_days',
        'next_order_date', 'churn_90d'],
       dtype='str'))

In [ ]:
df = df.sort_values('order_purchase_timestamp').reset_index(drop=True)
df['late_delivery'].value_counts()

late_delivery
0    1687
1      96
Name: count, dtype: int64

In [ ]:
print(f"{df['prior_total_spend'].sum().round(3)} total spend among returning customers")

288538.14 total spend among returning customers


In [ ]:
model_df = df.copy()

model_df = model_df[
    [
        'order_id',
        'customer_unique_id',
        'delivery_delay_days',
        'review_score',
        'payment_value',
        'num_prior_orders',
        'days_since_last_order',
        'customer_tenure_days',
        'prior_total_spend',
        'avg_prior_order_value',
        'churn_90d'
    ]
].copy()

model_df.to_parquet('../data/processed/model_df.parquet', index=False)